# ACDC / MRzero: reproducible weekly research figures

**Presentation:** 11 November 2026, 15 selected slides (11 main, 4 backups).

**Problem → Approach → Result.** This notebook quantifies measured OFF-model fidelity, tests numerical explanations for the structured intensity pattern, and separates implementation parity from scanner validation. It generates every data-driven slide figure from measured data, production code, or explicitly identified archived numerical arrays. It never extracts figures from PowerPoint.

The production 08, 08 V2 and 09 notebooks and physical model are unchanged. **Current 08 V2 uses a matrix DFT inverse**; the forced direct Fourier adjoint is a separate controlled comparator. “ADC coordinates” here are calculated from nominal Pulseq gradients, not independently measured scanner trajectories.

Run cells in order using the project scientific Python environment. Default execution regenerates figures from validated numerical caches. Set `RUN_EXPENSIVE=True` to rerun full simulations, controlled diagnostics and measured CPU benchmarks. Set `REFRESH_INPUTS=True` to also repeat the full 32-echo, 60-iteration fit. All result caches are checked against input and prepared-source hashes before reuse.

**Slide 8 revision:** compare a genuine centered nominal-grid IFFT directly with the current 08 V2 matrix inverse on exactly the same saved complex k-space. Do not relabel the automatic Pulseq-coordinate operator as IFFT. The automatic/direct-adjoint test remains supplementary; downstream production simulations and 08/09 equivalence retain their original reconstruction.


In [ ]:
from pathlib import Path
import sys, os, json, csv, numpy as np
from IPython.display import display, Image, Markdown

# Resolve from this folder, its parent, or the project root; never silently pick another project.
_candidates = [Path.cwd(), Path.cwd()/"presentation", Path.cwd()/"MRzero_generations_08_09"/"presentation"]
PRESENTATION = next((p.resolve() for p in _candidates if (p/"prepare_data.py").is_file()), None)
if PRESENTATION is None:
    raise FileNotFoundError("Start Jupyter in presentation/, MRzero_generations_08_09/, or the project root.")
sys.path.insert(0, str(PRESENTATION))
sys.path.insert(1, str(PRESENTATION.parent))

RUN_EXPENSIVE = False
RUN_FAST = not RUN_EXPENSIVE
REFRESH_INPUTS = False
THREADS = 2
import torch
torch.set_num_threads(THREADS)
from prepare_data import prepare_context, safe
from generate_figures import (Figures, validate_controlled_results, main_figures,
    geometry_figures, controlled_figures, sampling_controls_figure, benchmark_figures, coherence_figure,
    input_map_figures, local_q_figures, optimization_replay_figure, add_receiver_and_legacy, export_results)
print("Scientific interpreter:", sys.executable)
print("Presentation folder:", PRESENTATION)
print(dict(RUN_FAST=RUN_FAST, RUN_EXPENSIVE=RUN_EXPENSIVE, REFRESH_INPUTS=REFRESH_INPUTS))

def show_figure(alias, width=1250):
    display(Image(filename=str(PRESENTATION/"figures"/(alias+".png")), width=width))

def show_rows(rows, columns=None):
    if not rows:
        return
    columns = columns or list(rows[0])
    def cell(v):
        return f"{v:.6g}" if isinstance(v, (float, np.floating)) else str(v)
    display(Markdown("| " + " | ".join(columns) + " |\n|" + "|".join(["---"]*len(columns)) + "|\n" +
                     "\n".join("| " + " | ".join(cell(row.get(k, "")) for k in columns) + " |" for row in rows)))

## Inputs, fit, hardware and provenance — slides 4, 14, 15

`prepare_context` executes only the **current Generation 08** import/configuration, metadata, full 32-echo fit and finite-voxel preparation cells. Executed source cells and their hashes are saved in `cache/executed_preparation_cells.json`. CPU/precision and output/display flags are the only execution overrides; fit options and physical assumptions are retained.

The source object is an effective RSS-derived amplitude with residual exponential relaxation and unresolved coherence. It is not calibrated proton density or intrinsic T2. MRI receiver channels and the 32 ACDC shim outputs are distinct. The full preparation is cached; input-file hashes and code hashes are checked on reuse. Git SHA alone is insufficient because the generation sources are untracked/dirty.

In [ ]:
context = prepare_context(refresh=REFRESH_INPUTS, threads=THREADS)
registry = Figures(context)
summary = {}
show_rows([{"input":key,"path":value,"sha256":context["input_sha256"][key]} for key,value in context["source_paths"].items()])
print("Git revision:", context["provenance"]["git_sha"])
print(context["provenance"]["git_note"])
print("Fit policy:", context["fit_policy"])
print("Preparation seconds:", context["timings"])
print("Current limits [A]:", context["max_current_A"])
print("Total absolute-current limit [A]:",context["total_current_limit_A"])
print("Field constraint [Hz]:",context["field_limit_Hz"])
print(json.dumps(safe(context["prepared"]["report"]), indent=2))

## Fresh calculations or verified cache reuse — slides 3, 8, 10–14

**Problem:** comparisons must use identical objects and k-space, and reported runtimes must be measured.

**Approach:** the expensive runner first measures three fresh model constructions per generation, then one warm-up and three repetitions of forward/reconstruction, reconstruction-only and loss/gradient. It uses one prepared object, the same CPU, float64 operators (native MRzero float32), and two Torch threads. Library imports, input loading and the multi-echo refit are outside model timing. Cold construction and first zero-current forward are reported separately. Reconstruction comparators use exactly the saved Generation-08 complex signal.

It then computes zero-current and bounded deterministic dynamic-current 08/09 comparisons, equal-amplitude finite-support B0 controls, a separate rectangular ADC-aperture diagnostic, saved-current replay when available, and local q maps. Diagnostic currents are not a new optimization. These controls do not modify production physics.

In [ ]:
if RUN_EXPENSIVE:
    from controlled_experiments import run_controlled_experiments
    run_controlled_experiments(context, PRESENTATION/"data", run_expensive=True,
                              warmup=1, repeats=3, setup_repeats=3)
else:
    print("RUN_FAST: using recorded numerical calculations; all figures are regenerated below.")
validate_controlled_results(context)
print("Controlled-result input and prepared-source hashes verified.")

## Geometry, nominal k and ADC time — slides 4, 13

The common physical frame uses the exact production frame matrix. K coordinates are cycles/m; displayed X and Y are mm. Every stored ADC retains its RF-relative timestamp after odd/even readout reversal and row sorting. Navigator samples are handled by the original acquisition reader; no additional samples are discarded here. Gradient-derived trajectory agreement does not establish agreement with the scanner's actual gradients.

In [ ]:
summary["geometry"] = geometry_figures(context, registry)
show_figure("slide13_trajectory_timing")
print(json.dumps(safe(summary["geometry"]), indent=2))

## Measured OFF fidelity and shared reconstruction controls — slides 3, 8, 10

**Problem:** validate OFF before interpreting current optimization.

**Approach:** all primary OFF metrics use all 16,384 finite FOV pixels, no registration, and the positive scalar least-squares magnitude gain
\[
g=\frac{\sum_{m\in\Omega}M_m|I_m|}{\sum_{m\in\Omega}|I_m|^2},\qquad
\mathrm{NRMSE}=100\frac{\|g|I|-M\|_{2,\Omega}}{\|M\|_{2,\Omega}}.
\]
The gain from automatic Generation 08 is held fixed for every reconstruction comparator. Slide 8 compares **the true centered IFFT**
\[
I_{\mathrm{IFFT}}=\mathrm{fftshift}\!\left[\mathrm{ifft2}\!\left(\mathrm{ifftshift}(s)\right)\right]
\]
with the production **08 V2 matrix inverse**, \(I_{\mathrm{matrix}}=A^+s\), \(\lambda=0\). Both consume exactly the same complex simulated k-space, with unchanged storage order, physical output grid, Fourier sign, \(1/N\) scale and fixed reporting gain. IFFT deliberately assigns the stored samples to centered nominal Cartesian bins; the matrix uses unrounded Pulseq-calculated ADC coordinates. The latter are not independently measured scanner trajectories.

The asymmetric exactly Cartesian synthetic control now explicitly checks **IFFT versus matrix inverse**, as well as the known complex object. Automatic actual-coordinate row separation versus direct adjoint remains a supplementary numerical check. Direct adjoint is not a pseudoinverse on a general non-Cartesian grid.

GRE is resampled using production physical geometry. Its coverage mask and reference scaling are separate from full-FOV OFF NRMSE. The shared patient figures are computed together to guarantee identical underlying arrays.

In [ ]:
patient, X_m, Y_m = main_figures(context, registry)
summary.update(patient)
show_figure("slide03_off_validation")
print("Fresh uncalibrated full-FOV OFF metrics:", summary["off"])
show_figure("slide10_gre_off_comparison")
show_figure("matched_generations_off")


### Same-signal IFFT versus matrix-inverse reconstruction — slide 8

**Problem:** does replacing IFFT with the matrix inverse change the apparent intensity pattern?

**Approach:** use the same complex k-space and one fixed Gen08 reporting gain, without fitted complex rescaling or phase alignment. Show IFFT, matrix inverse, **absolute complex difference**, and **absolute magnitude difference** separately. Their colorbars have distinct, explicit scales.
\[
\epsilon_{\mathrm{complex}}=\frac{\|I_{\mathrm{matrix}}-I_{\mathrm{IFFT}}\|_2}{\|I_{\mathrm{matrix}}\|_2},\qquad
\epsilon_{\mathrm{magnitude}}=100\frac{\||I_{\mathrm{matrix}}|-|I_{\mathrm{IFFT}}|\|_2}{\||I_{\mathrm{matrix}}|\|_2}.
\]

**Result:** recomputed values below distinguish a phase-sensitive complex difference from changes in magnitude. The conclusion is limited to the supplied Pulseq trajectory. The production 08/09 forward-model comparison is kept separate, using its original matched automatic reconstruction to isolate implementation differences.

In [ ]:
show_figure("slide08_reconstruction_control")
pair = summary["reconstruction"]["ifft_vs_matrix"]
show_rows([pair], ["complex_relative_image_error", "magnitude_difference_NRMSE_percent",
                  "IFFT_OFF_NRMSE_percent", "matrix_OFF_NRMSE_percent",
                  "Cartesian_IFFT_vs_matrix_relative_L2"])
print("Pair conventions and phase diagnostic:", pair)
print("Exactly Cartesian sanity:", summary["reconstruction"]["sanity"])
show_figure("reconstruction_auto_vs_direct")
show_figure("reconstruction_forced_grid")
show_rows(summary["reconstruction"]["rows"], ["operator","complex_relative_image_error","magnitude_difference_NRMSE_percent","OFF_NRMSE_percent"])
print("Supplementary fixed-mask pattern metrics:", [(r["operator"],r["pattern"]) for r in summary["reconstruction"]["rows"]])

## 08/09 implementation equivalence, uniform object and ADC aperture — slides 2, 9, 11, 12, 14

08 recomputes current-dependent phase and three sinc factors around cached native center histories. 09 calls the locally modified `native_pdg.execute_graph` with dynamic B0 and local k inputs. Both retain the same supported physics and one uniform receiver. Residual unresolved coherence is applied once.

We compare every complex ADC and the matched complex image for zero and nonzero deterministic currents:
\[\epsilon_s=\|s_{09}-s_{08}\|_2/\|s_{08}\|_2.\]

The uniform diagnostic sets retained-voxel amplitude to one, exponential relaxation to infinity and unresolved dispersion to none, holds currents at zero, and changes only measured B0/slopes versus zero B0/slopes. The fixed interior mask is the retained support eroded three times with SciPy's default connectivity. CV is standard deviation/mean magnitude; adjacent RMS is the RMS nearest-neighbor magnitude difference/mean. Finite support and reconstruction edges still matter.

The ADC diagnostic averages a **hypothetical rectangular 5 μs aperture**, using 3- and 5-node Gauss quadrature and Pulseq gradient antiderivatives. The center oracle is checked against the production native signal first. Scanner receiver-filter information is unavailable: this is a controlled sensitivity test, not a demonstrated scanner correction.

In [ ]:
summary["equivalence"], summary["uniform_phantom"], summary["ADC_window"] = controlled_figures(context, registry)
show_figure("slide11_signal_equivalence", width=900)
show_rows(summary["equivalence"]["rows"], ["case","signal_complex_relative_L2","image_complex_relative_L2","image_magnitude_NRMSE_percent"])
show_figure("uniform_phantom_b0_control")
print(json.dumps(safe(summary["uniform_phantom"]), indent=2))
show_figure("slide12_adc_window")
print(json.dumps(safe(summary["ADC_window"]), indent=2))
sampling_controls_figure(context, registry)
show_figure("slide12_sampling_controls")


## Measured setup and iteration runtimes — slide 11

Points are measured medians and whiskers are the interquartile range, not timing estimates. Full raw repetitions, warm-up count, CPU identity, thread count, dtypes and exact inclusions are stored in `data/runtime_benchmarks.json` and `.csv`. This is a shared CPU host, not a GPU benchmark or an exclusive-host claim. Setup time is distinct from recurring operations. Current projection and field-guidance evaluation are outside the timed common magnitude-loss/gradient operation.

In [ ]:
summary["runtime"] = benchmark_figures(registry)
show_figure("slide11_runtime_benchmarks")
show_rows(summary["runtime"]["records"], ["generation","operation","median_seconds","IQR_seconds","warmup_count","repetitions"])
print("Hardware:", summary["runtime"]["provenance"])
print("Common settings:", summary["runtime"]["common_settings"])

## Local voxel coherence and q maps — slides 5–7, 14

For internal normalized coordinates \(u_a\in[-1/2,1/2]\),
\[
q_{a,v,n}=\mathbf{k}_n\cdot\mathbf{e}_{a,v}+g_{B0,a,v}t_n+
\sum_jg_{ACDC,a,v,j}J_{j,n},\qquad
F_{v,n}=\int_{[-1/2,1/2]^3}e^{-i2\pi\sum_a q_{a,v,n}u_a}\,d^3u
=\prod_{a=1}^{3}\operatorname{sinc}(q_{a,v,n}).
\]
\(q\) is cycles **across that voxel**, not phase and not a spatial constant. Edges are in meters, B0 slopes in Hz/full voxel, calibrated ACDC slopes in Hz/A/full voxel, and integrated currents in A·s. The normal B0 slope is unmeasured and assumed zero; normal ACDC slopes come from 3D calibration. The scalar fitted residual-coherence factor already resides in the point-center magnetization and is not applied again here.

The first figure checks the theoretical integral by numerical quadrature. The maps use actual prepared per-voxel slopes and bounded diagnostic currents, at the ADC nearest k=0; first/last ADC maps are also saved in `local_q.npz`.

In [ ]:
summary["coherence_integral"] = coherence_figure(context, registry)
summary["local_q"] = local_q_figures(context, registry)
summary["input_maps"] = input_map_figures(context, registry)
show_figure("slide07_sinc_integral_check", width=1000)
show_figure("local_q_maps")
show_figure("input_maps")
print("Integral check:",summary["coherence_integral"])

## Legacy pattern and actual receive implementation — slides 2, 4, 6, 14

The legacy figure is regenerated from the exact archived Generation 06 V3 numerical output identified by the original notebook, with its seven input hashes checked. Its original source-code hashes are unavailable, so it is an archived numerical diagnostic, not a rerun of an exact historical implementation. The legacy B0=0 visual observation is retained as a hypothesis; CV and adjacent variation do not establish that B0=0 removes all structure.

The receiver audit reads active physical channel IDs from Twix headers and traces per-receiver complex echo fitting through to EPI forward simulation. It distinguishes measured coil images from calibrated sensitivity maps. If the current EPI code still emits one uniform-channel signal, no before/after receiver improvement is claimed. A separate spiral-only receive simulator does not imply EPI support. Effective fitted scalar amplitude already includes receive magnitude; adding unnormalized coil amplitudes would double-count it.

In [ ]:
summary["legacy"], summary["receiver_audit"] = add_receiver_and_legacy(context, registry)
show_figure("slide02_legacy_pattern")
show_figure("slide04_receiver_channels")
print(json.dumps(safe(summary["receiver_audit"]), indent=2))

## Archived current replay, if present — numerical evidence only

Saved raw and DAC-quantized commands are replayed through the current prepared model, without rerunning optimization or changing the commands. The scanner ON/OFF mapping and physical improvement are not established by this simulated replay. This result remains separate from primary OFF fidelity and implementation parity; no removed optimization slide is restored.

In [ ]:
summary["optimization_replay"] = optimization_replay_figure(context, registry)
if summary["optimization_replay"] is not None:
    show_figure("archived_current_replay")
    print(json.dumps(summary["optimization_replay"], indent=2))
else:
    print("No verified saved-current replay is available. RUN_EXPENSIVE=True will attempt the documented archive path.")

## Export scientific data, figure manifest and slide equation sources — slide 15

Exports include clean PNG at 300 dpi plus SVG/PDF, exact numerical `.npz`, measured runtime CSV, consolidated `results_metrics.csv`, source/input hashes, per-figure metadata and `presentation_summary.json`. Masks, gains, normalization, physical coordinates, device and timestamp accompany results.

Scientific diagrams remain editable native shapes in the PowerPoint. Equations use vector SVG generated by genuine LaTeX, with high-resolution PNG compatibility previews because this environment's Office-Math conversion did not render reliably. Every literal LaTeX source is retained in `equations/`, `equations_manifest.json`, and slide speaker notes. They can be regenerated without hand transcription.

In [ ]:
summary, manifest = export_results(context, registry, summary)
show_rows([{"figure":name,"slides":info["slides"],"PNG":info["files"]["png"]} for name,info in manifest["figures"].items()])
equations = PRESENTATION/"equations_manifest.json"
if equations.exists():
    display(Markdown("Literal LaTeX equation provenance: `equations_manifest.json`; vector and source files: `equations/`."))
print("Metrics:", PRESENTATION/"results_metrics.csv")
print("Figure manifest:", PRESENTATION/"figures_manifest.json")
print("Summary:", PRESENTATION/"presentation_summary.json")
from export_slide_data import export_slide_data
export_slide_data()
print("Portable deck bridge written: presentation_slide_data.json")


## Rebuild the editable deck and verify it

After numerical generation, run the separate builder from `presentation/`:

```bash
PYTHONPATH=/tmp/presentation_deps python3 build_presentation.py
```

The builder loads the user's selected 15-slide source deck and consumes these exported results. `README.md` documents the dependency setup, precise render command, visual audit, slide changes, remaining hypotheses, and full rerun commands. Rebuilding the PowerPoint does not rerun the physical calculations. Figures are replaceable embedded images, while text, physical diagrams and arrows remain native editable slide elements.